In [0]:
import json
import time
import requests
from azure.eventhub import EventHubProducerClient, EventData

URL = spark.conf.get("lab5.streaming.source_url")

secret_scope = spark.conf.get("lab5.eventhub_secret_scope")
producer_connstr = spark.conf.get("lab5.eventhub_producer_connstr_key")
eventhub_name = spark.conf.get("lab5.eventhub_name_key")

EVENT_HUB_CONNECTION_STRING = dbutils.secrets.get(
    scope=secret_scope,
    key=producer_connstr
)

EVENT_HUB_NAME = dbutils.secrets.get(
    scope=secret_scope,
    key=eventhub_name
)

producer = EventHubProducerClient.from_connection_string(
    conn_str=EVENT_HUB_CONNECTION_STRING,
    eventhub_name=EVENT_HUB_NAME
)

try:
    seen_ids = set()
    while True:
        data = requests.get(URL).json()

        batch = producer.create_batch()

        for earthquake in data["features"]:

            earthquake_id = earthquake["id"]

            if earthquake_id in seen_ids:
                continue

            seen_ids.add(earthquake_id)

            batch.add(
                EventData(
                    json.dumps(earthquake)
                )
            )

        producer.send_batch(batch)

        print(f"Sent {len(data['features'])} earthquakes")

        time.sleep(5)

finally:
    producer.close()